# 23. Recurrent Neural Network (RNN)
**Use case:** Oil & Gas — Daily oil production rate forecasting for a well

An operator needs short-term production forecasts for each well to plan trucking, sales nominations and field crew visits. Rates decline over time, but noise, shut-ins and choke changes make simple decline curves miss short-term behaviour. The SCADA historian holds daily oil rates, shut-in flags and well age. Success means forecasts for the next 30 days that beat the decline-curve and naive baselines on producing days, plus a clear flag when actual production falls well below forecast.

> 💡 It is like a cricket commentator who keeps a running sense of the innings: after each ball, they update their mental summary of momentum and use it to predict the next over. They do not replay every ball; they carry a compressed memory forward.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Watch an RNN remember 5 days of oil
Compute an RNN's hidden state by hand in numpy over 5 days of a well's oil rate and see how memory carries forward.

| 5W + H | |
|---|---|
| **Who** | A production engineer curious how a neural network can 'remember' yesterday's rate. |
| **What** | A 3-unit RNN cell applied to 5 daily oil rates, printing the hidden state after each day. |
| **When** | Each new day's rate is fed in as it arrives, updating the network's memory. |
| **Where** | One producing well, using plain numpy with no deep-learning library. |
| **Why** | Seeing the hidden-state update makes LSTMs, GRUs and PyTorch's nn.RNN much less mysterious. |
| **How** | h_t = tanh(W_xh x_t + W_hh h_(t-1) + b), with the same weights reused every day, and a least-squares output layer to show learning. |

**🧠 ML concepts:** Hidden state as memory, Weight sharing across time steps, tanh activation, Input scaling, Fitting an output layer  
**🐍 Python concepts:** numpy arrays and division by a scalar, Matrix multiplication with @, enumerate with start=1, Generator expressions inside join, Star-unpacking (W_fit, *_), .item() and .ravel()

In [ ]:
# RNN by hand: watch the hidden state "remember" 5 days of oil production for one well
import numpy as np

np.random.seed(42)
# 🐍 PY: a numpy array of 5 daily oil rates in barrels per day (bbl/d); the well is slowly declining
oil_bbl_per_day = np.array([520.0, 505.0, 498.0, 470.0, 462.0])
# 🧠 ML: neural nets learn better on small numbers, so scale rates to roughly 0..1
x = oil_bbl_per_day / 600.0

HIDDEN = 3
# 🧠 ML: the RNN has just three weight sets, REUSED at every day (that's what "recurrent" means)
# 🐍 PY: np.random.randn(rows, cols) creates a matrix of random normal numbers
W_xh = np.random.randn(HIDDEN, 1) * 0.5      # input -> hidden
W_hh = np.random.randn(HIDDEN, HIDDEN) * 0.5  # previous hidden -> hidden (the "memory" path)
W_hy = np.random.randn(1, HIDDEN) * 0.5      # hidden -> prediction
b_h = np.zeros((HIDDEN, 1))

# 🧠 ML: hidden state h starts empty - the network knows nothing before day 1
h = np.zeros((HIDDEN, 1))
print("day | rate_bbl/d | hidden state h (3 numbers)      | raw output")
# 🐍 PY: enumerate(..., start=1) numbers the days from 1 instead of 0
for day, xt in enumerate(x, start=1):
    # 🐍 PY: @ is matrix multiplication; np.array([[xt]]) makes a 1x1 column so shapes line up
    # 🧠 ML: new memory = tanh(today's input + yesterday's memory); tanh squashes into -1..1
    h = np.tanh(W_xh @ np.array([[xt]]) + W_hh @ h + b_h)
    y = (W_hy @ h).item()
    # 🐍 PY: .ravel() flattens the 3x1 column to a simple list for printing
    h_txt = " ".join(f"{v:+.3f}" for v in h.ravel())
    print(f" {day}  |   {oil_bbl_per_day[day - 1]:6.1f}   | [{h_txt}]          | {y:+.3f}")

# 🧠 ML: after day 5, h is a compressed summary of all 5 days - that's what we forecast from
print("\nFinal hidden state summarises the whole 5-day history:", np.round(h.ravel(), 3))

# 🧠 ML: the weights are untrained, so the raw output is meaningless until we fit W_hy
# Quick 'training' of only the output layer with least squares, to show the idea of learning:
H_all, h = [], np.zeros((HIDDEN, 1))
for xt in x[:-1]:
    h = np.tanh(W_xh @ np.array([[xt]]) + W_hh @ h + b_h)
    H_all.append(h.ravel())
# 🐍 PY: np.linalg.lstsq solves "find W so that H_all @ W ~= next-day targets"
W_fit, *_ = np.linalg.lstsq(np.array(H_all), x[1:], rcond=None)
pred = np.array(H_all) @ W_fit * 600
# 🧠 ML: caution - 3 weights fit to 4 points, in-sample; real training needs far more days (see Medium)
for d, (p, a) in enumerate(zip(pred, oil_bbl_per_day[1:]), start=2):
    print(f"Day {d}: predicted {p:6.1f} bbl/d   actual {a:6.1f}")

**Try this next:**
- Multiply W_hh by 3 and watch the hidden values saturate near +1 or -1.
- Set W_hh to zeros: the RNN now has no memory. What changes in the printout?
- Add a shut-in day (rate 0) in the middle and watch the hidden state react.

## MEDIUM: PyTorch RNN on 3 years of well data
Train a PyTorch nn.RNN on sliding 14-day windows to forecast tomorrow's oil rate and compare it with a naive baseline.

| 5W + H | |
|---|---|
| **Who** | A production surveillance analyst forecasting next-day rates to spot under-performing days. |
| **What** | A 16-unit RNN that reads the last 14 days of rate and predicts tomorrow, trained on 3 years of synthetic data. |
| **When** | Daily, after the morning production report is loaded. |
| **Where** | One well whose rate follows a hyperbolic decline with noise and random shut-ins, in PyTorch on CPU. |
| **Why** | A forecast gives an 'expected' rate; days far below it can be flagged for a pumper or engineer to check. |
| **How** | Scale with train-period stats, build sliding windows, split by time, train with Huber loss and gradient clipping, then compare with 'same as yesterday'. |

**🧠 ML concepts:** Sliding windows for sequence data, Time-ordered train/test split, Scaling without leakage, Residual prediction, Huber loss for outliers, Gradient clipping  
**🐍 Python concepts:** Classes inheriting from nn.Module, super().__init__(), List comprehension with np.stack, Adding a dimension with [..., None], torch.no_grad() context manager, Boolean masks

In [ ]:
# PyTorch nn.RNN: forecast tomorrow's oil rate for one well from the last 14 days
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)

# ---- Synthetic 3 years of daily production: hyperbolic decline + noise + shut-ins ----
days = np.arange(1095)
qi, Di, b = 900.0, 0.004, 0.8
# 🧠 ML: Arps hyperbolic decline q(t) = qi / (1 + b*Di*t)^(1/b) - the classic petroleum engineering curve
base = qi / (1 + b * Di * days) ** (1 / b)
noise = np.random.normal(0, 0.04, len(days)) * base
oil = base + noise
# Random shut-ins (workovers, pump failures): rate drops to ~0 for 2-5 days
for start in np.random.choice(len(days) - 6, 12, replace=False):
    oil[start:start + np.random.randint(2, 6)] = np.random.uniform(0, 20)
# 🐍 PY: np.clip keeps values in a valid range (no negative barrels)
df = pd.DataFrame({"day": days, "oil_bbl_d": np.clip(oil, 0, None)})
print(df.describe().loc[["mean", "min", "max"]].round(1).T)

# 🧠 ML: scale using the TRAIN period only, to avoid leaking future information
SPLIT = int(len(df) * 0.8)
scale = df.oil_bbl_d[:SPLIT].max()
series = (df.oil_bbl_d / scale).to_numpy(dtype=np.float32)

WINDOW = 14


def make_windows(arr, window):
    # 🧠 ML: sliding windows turn a time series into (last 14 days -> next day) training examples
    # 🐍 PY: list comprehension + np.stack builds a 2-D array of overlapping slices
    X = np.stack([arr[i:i + window] for i in range(len(arr) - window)])
    y = arr[window:]
    return X, y


X, y = make_windows(series, WINDOW)
split_idx = SPLIT - WINDOW
# 🧠 ML: time-ordered split - train on the past, test on the future (never shuffle across time)
# 🐍 PY: [..., None] adds a trailing feature dimension: shape becomes (samples, 14, 1)
X_tr, X_te = torch.tensor(X[:split_idx, :, None]), torch.tensor(X[split_idx:, :, None])
y_tr, y_te = torch.tensor(y[:split_idx]), torch.tensor(y[split_idx:])
print(f"\nTrain windows: {len(X_tr)}  Test windows: {len(X_te)}")


class OilRNN(nn.Module):
    # 🐍 PY: a class that inherits from nn.Module; __init__ defines layers, forward defines the math
    def __init__(self, hidden=16):
        super().__init__()
        # 🧠 ML: nn.RNN runs h_t = tanh(W_x x_t + W_h h_{t-1} + b) across all 14 days for us
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, h_last = self.rnn(x)
        # 🧠 ML: predict the CHANGE from the last observed day (residual), from the 14-day summary
        # 🧠 ML: this makes "same as yesterday" the starting point the RNN only has to improve on
        return x[:, -1, 0] + self.head(h_last[-1]).squeeze(-1)


model = OilRNN()
opt = torch.optim.Adam(model.parameters(), lr=0.005)
# 🧠 ML: Huber loss = MSE for small errors, MAE for big ones, so shut-in days don't dominate training
loss_fn = nn.HuberLoss(delta=0.02)
for epoch in range(1, 41):
    model.train()
    # 🐍 PY: torch.randperm shuffles window order (fine WITHIN train - each window keeps its own order)
    perm = torch.randperm(len(X_tr))
    for i in range(0, len(perm), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = loss_fn(model(X_tr[idx]), y_tr[idx])
        # 🧠 ML: backpropagation through time - gradients flow back across all 14 steps
        loss.backward()
        # 🧠 ML: gradient clipping stops rare "exploding" updates that RNNs are prone to
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    if epoch % 10 == 0:
        print(f"epoch {epoch:2d}  train Huber loss={loss.item():.5f}")

model.eval()
# 🐍 PY: torch.no_grad() turns off gradient tracking for faster, memory-light inference
with torch.no_grad():
    pred = model(X_te).numpy() * scale
actual = y_te.numpy() * scale
naive = X_te[:, -1, 0].numpy() * scale
mae = np.mean(np.abs(pred - actual))
naive_mae = np.mean(np.abs(naive - actual))
print(f"\nTest MAE  RNN: {mae:5.1f} bbl/d   naive 'same as yesterday': {naive_mae:5.1f} bbl/d")
# 🧠 ML: shut-ins are unpredictable from rates alone; score "normal" days separately (both sides > 50 bbl/d)
normal = (actual > 50) & (naive > 50)
print(f"Normal days only ({normal.sum()}): RNN {np.mean(np.abs(pred - actual)[normal]):5.1f}"
      f"  naive {np.mean(np.abs(naive - actual)[normal]):5.1f} bbl/d")
print("\nLast 5 test days:")
for p, a in zip(pred[-5:], actual[-5:]):
    print(f"  actual {a:6.1f}  predicted {p:6.1f}")
print(f"\nBusiness read-out: mean test rate {actual.mean():.0f} bbl/d; RNN error {mae:.0f} bbl/d"
      f" ({100 * mae / actual.mean():.1f}%). For a ONE-day horizon it only ties the naive rule -")
print("so always report a naive baseline. The value of sequence models shows up in multi-day forecasts (Hard).")

**Try this next:**
- Change WINDOW to 30 and see if test MAE improves.
- Remove the residual (return the head output directly) and compare MAE.
- Swap nn.RNN for nn.GRU (same arguments) and rerun.

## HARD: 30-day forecast: RNN vs Arps decline curve
Forecast the next 30 days with an RNN and a GRU, compare them with an Arps decline fit and a naive baseline, and measure vanishing gradients.

| 5W + H | |
|---|---|
| **Who** | An asset team's production engineer and data scientist building a short-term forecast for operations planning. |
| **What** | A direct multi-step RNN forecaster (30 days in, 30 days out) with shut-in flags and well age as inputs, benchmarked against scipy-fitted Arps. |
| **When** | Every two weeks for the monthly operations plan, and on demand after interventions such as choke changes. |
| **Where** | One well with 4 years of daily data including a choke cut, run in PyTorch and scipy on CPU. |
| **Why** | Arps curves are trusted for reserves but react slowly to operational changes; a sequence model re-anchors to the latest behaviour. |
| **How** | Strict time split, per-window normalisation, masked loss that ignores future shut-ins, MAPE by horizon, a gradient-by-lag check, and saving weights. |

**🧠 ML concepts:** Direct multi-step forecasting, Physics-based baseline (Arps), Masked loss for missing or outage days, Per-window normalisation, Vanishing gradients in plain RNNs, LSTM/GRU as gated upgrades  
**🐍 Python concepts:** Dictionaries mapping names to classes, Lambda as a small helper, Boolean masks with ~, requires_grad and .grad for sensitivity, torch.save of a state_dict, scipy curve_fit with bounds

In [ ]:
# Production-style 30-day oil-rate forecast for one well: RNN vs Arps decline curve (scipy),
# time-ordered split, shut-in handling, multi-step direct forecast, vanishing-gradient check, model save.
import os
import tempfile
import warnings
import numpy as np
import torch
import torch.nn as nn
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore")
torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)

WINDOW, HORIZON = 30, 30


def simulate_well(n_days=1460):
    """Hyperbolic decline + multiplicative noise + shut-ins + one choke change (operational step)."""
    t = np.arange(n_days, dtype=float)
    q = 1000 / (1 + 0.8 * 0.004 * t) ** (1 / 0.8)
    q *= np.where(t > 900, 0.85, 1.0)             # choke cut at day 900 -> 15% rate drop
    q *= 1 + rng.normal(0, 0.04, n_days)
    shut = np.zeros(n_days, bool)
    for s in rng.choice(n_days - 6, 15, replace=False):
        shut[s:s + rng.integers(2, 6)] = True
    q[shut] = rng.uniform(0, 15, shut.sum())
    return t, np.clip(q, 0, None), shut


def arps(t, qi, di, b):
    # 🧠 ML: Arps hyperbolic decline - 3 physical parameters, the industry's reserves-booking workhorse
    return qi / (1 + b * di * t) ** (1 / b)


def fit_arps(t, q, shut):
    # 🧠 ML: fit only producing days; shut-ins are operations events, not reservoir behaviour
    ok = ~shut
    # 🐍 PY: curve_fit returns (best params, covariance); bounds keep b in the physical 0..2 range
    p, _ = curve_fit(arps, t[ok], q[ok], p0=[q[ok][:30].mean(), 0.01, 0.5],
                     bounds=([1, 1e-5, 0.01], [5000, 1, 2.0]), maxfev=20000)
    return p


class RateRNN(nn.Module):
    """Input: 30 days of (scaled rate, shut-in flag, scaled well age). Output: next 30 days at once."""
    def __init__(self, n_feat=3, hidden=24, kind="rnn"):
        super().__init__()
        # 🧠 ML: swap "rnn" for "gru"/"lstm" - gated cells fix vanishing gradients on long windows
        cls = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.rnn = cls(n_feat, hidden, batch_first=True)
        self.head = nn.Linear(hidden, HORIZON)

    def forward(self, x):
        out, _ = self.rnn(x)
        # 🧠 ML: direct multi-step - one head emits all 30 future days, avoiding error snowballing
        # 🧠 ML: predict multipliers of the last 7-day producing level so output scale is stable
        return self.head(out[:, -1]).exp()


def make_xy(q, shut, t, idx_list):
    X, Y, L = [], [], []
    for i in idx_list:
        hist_q, hist_s = q[i - WINDOW:i], shut[i - WINDOW:i]
        # 🐍 PY: boolean mask ~hist_s picks producing days only; [-7:] takes the last 7 of them
        level = max(hist_q[~hist_s][-7:].mean(), 1.0)
        feats = np.stack([hist_q / level, hist_s.astype(float), t[i - WINDOW:i] / 1460], axis=1)
        X.append(feats)
        Y.append(q[i:i + HORIZON] / level)
        L.append(level)
    f = lambda a: torch.tensor(np.array(a), dtype=torch.float32)
    return f(X), f(Y), np.array(L)


def train(model, X, Y, Ymask, epochs=60, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        perm = torch.randperm(len(X))
        for b in range(0, len(X), 64):
            i = perm[b:b + 64]
            opt.zero_grad()
            # 🧠 ML: masked loss - ignore future shut-in days; they are unforecastable outages
            err = (model(X[i]) - Y[i]) ** 2 * Ymask[i]
            loss = err.sum() / Ymask[i].sum()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    return model


def grad_by_lag(model, X):
    """Vanishing-gradient check: how much does the forecast depend on each input day?"""
    x = X[:128].clone().requires_grad_(True)
    model(x).sum().backward()
    # 🐍 PY: .abs().mean(dim=(0, 2)) averages over samples and features, leaving one value per day
    g = x.grad.abs().mean(dim=(0, 2)).numpy()
    return g / g[-1]


if __name__ == "__main__":
    t, q, shut = simulate_well()
    cut = 1100
    # 🧠 ML: strict time split - train windows must END before the cutoff, test windows start after it
    train_idx = list(range(WINDOW, cut - HORIZON, 2))
    test_idx = list(range(cut, len(q) - HORIZON, 15))
    Xtr, Ytr, _ = make_xy(q, shut, t, train_idx)
    Xte, Yte, Lte = make_xy(q, shut, t, test_idx)
    Mtr = torch.tensor(np.array([~shut[i:i + HORIZON] for i in train_idx]), dtype=torch.float32)
    Mte = np.array([~shut[i:i + HORIZON] for i in test_idx])
    print(f"Train windows {len(Xtr)} (days {WINDOW}-{cut})  |  test forecast origins {len(Xte)} (days {cut}+)")

    params = fit_arps(t[:cut], q[:cut], shut[:cut])
    print(f"Arps fit on train: qi={params[0]:.0f} bbl/d  Di={params[1]:.4f}/day  b={params[2]:.2f}")

    results = {}
    for kind in ("rnn", "gru"):
        torch.manual_seed(42)
        m = train(RateRNN(kind=kind), Xtr, Ytr, Mtr)
        m.eval()
        with torch.no_grad():
            results[kind] = (m(Xte).numpy() * Lte[:, None], m)

    actual = Yte.numpy() * Lte[:, None]
    arps_pred = np.array([arps(t[i:i + HORIZON], *params) for i in test_idx])
    naive = np.repeat(Lte[:, None], HORIZON, axis=1)

    def mape(p):
        # 🧠 ML: score only producing days (mask) - shut-ins would swamp any percentage metric
        return 100 * np.mean(np.abs(p - actual)[Mte] / actual[Mte])

    print("\n30-day forecast error on unseen future (MAPE, producing days):")
    print(f"  {'Model':28s} {'days 1-7':>9} {'days 8-30':>10} {'all':>6}")
    rows = [("Naive (last 7-day level)", naive), ("Arps decline (scipy)", arps_pred),
            ("RNN (nn.RNN)", results["rnn"][0]), ("GRU upgrade", results["gru"][0])]
    for name, p in rows:
        e = np.abs(p - actual) / np.maximum(actual, 1)
        early = 100 * e[:, :7][Mte[:, :7]].mean()
        late = 100 * e[:, 7:][Mte[:, 7:]].mean()
        print(f"  {name:28s} {early:8.1f}% {late:9.1f}% {mape(p):5.1f}%")

    # 🧠 ML: honest read - on a smooth decline the naive level is hard to beat; the RNN only edges it
    print("\nNote: naive is a strong baseline on a smooth decline - the RNN edges it, it does not crush it.")
    print("Why Arps struggles here: the choke cut at day 900 shifted the level; Arps fitted the whole")
    print("history and cannot 'see' it, while sequence models re-anchor on the latest 30 days.")

    g = grad_by_lag(results["rnn"][1], Xte)
    print("\nVanishing-gradient check (input sensitivity relative to the most recent day):")
    for lag in (1, 5, 10, 20, 30):
        print(f"  {lag:2d} days back: {g[-lag]:.3f} " + "#" * int(40 * min(g[-lag], 1)))
    print("  -> a plain RNN barely uses inputs from 3-4 weeks back; LSTM/GRU gates are designed to keep it.")

    print("\nModel-risk note: reserves (SEC/PRMS) still rely on Arps-type curves an engineer can defend;")
    print("use the RNN for short-term ops forecasting and anomaly flags, validated monthly vs actuals.")

    out = os.path.join(tempfile.mkdtemp(), "oil_rnn.pt")
    # 🐍 PY: state_dict() is a dict of weight tensors; saving it (not the object) is the PyTorch best practice
    torch.save({"rnn": results["rnn"][1].state_dict(), "arps": params.tolist()}, out)
    print(f"Saved RNN weights + Arps params -> {os.path.basename(out)} in a temp dir")

**Try this next:**
- Remove the choke cut (factor 1.0) and see whether Arps catches up with the RNN.
- Set WINDOW = 60 and rerun the gradient-by-lag check for RNN vs GRU.
- Add 'lstm' to the model loop and compare its MAPE.